# Part 2: Advanced RAG

Three advanced techniques on top of the naive baseline:

| Technique | What it does |
|---|---|
| **Parent-Child Chunking** | Small chunks (300 chars) for precise retrieval; return their large parent (1000 chars) to the LLM for context |
| **Hybrid Search (BM25 + Vector + RRF)** | BM25 catches exact keyword matches; vector search catches semantic matches; RRF fuses the two ranked lists |
| **Cross-Encoder Reranking** | Retrieve top-20, rescore every candidate with `bge-reranker-v2-m3` (multilingual), keep top-5 for generation |

In [ ]:
import json
import os
import re
import numpy as np
import faiss
import pypdf
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

INPUT_DIR   = Path("input")
PDF_PATH    = INPUT_DIR / "ilovepdf_merged.pdf"
QA_PATH     = INPUT_DIR / "q_a.json"
OUTPUT_PATH = Path("results_advanced.json")

CHILD_SIZE    = 300    # small chunks for retrieval
CHILD_OVERLAP = 50
PARENT_SIZE   = 1000   # large chunks sent to LLM
PARENT_OVERLAP = 200

RETRIEVE_K    = 20     # candidates before reranking
TOP_K         = 5      # final chunks sent to LLM
RRF_K         = 60     # RRF constant

EMBED_MODEL    = "text-embedding-3-small"
GEN_MODEL      = "gpt-4o-mini"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"  # multilingual (was: bge-reranker-base, English-only)

## 1. Load PDF

In [ ]:
def load_pdf(path: Path) -> list[dict]:
    reader = pypdf.PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page_num": i, "text": text})
    print(f"Loaded {len(pages)} pages from {path.name}")
    return pages

pages = load_pdf(PDF_PATH)

## 2. Technique 1 — Parent-Child Chunking

Each **child** chunk (small, used for retrieval) stores a pointer to its **parent** chunk (large, sent to the LLM).  
This gives precise matching on retrieval while providing rich context for generation.

In [ ]:
def sliding_chunks(text: str, size: int, overlap: int) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size].strip())
        start += size - overlap
    return [c for c in chunks if c]


def build_parent_child(pages: list[dict]) -> tuple[list[dict], list[dict]]:
    """
    Returns:
      parents: [{id, page_num, text}]
      children: [{id, parent_id, page_num, text}]
    """
    parents, children = [], []
    pid = 0
    cid = 0
    for page in pages:
        for p_text in sliding_chunks(page["text"], PARENT_SIZE, PARENT_OVERLAP):
            parents.append({"id": pid, "page_num": page["page_num"], "text": p_text})
            for c_text in sliding_chunks(p_text, CHILD_SIZE, CHILD_OVERLAP):
                children.append({"id": cid, "parent_id": pid, "page_num": page["page_num"], "text": c_text})
                cid += 1
            pid += 1
    return parents, children


parents, children = build_parent_child(pages)
print(f"Parents (sent to LLM): {len(parents)}")
print(f"Children (used for retrieval): {len(children)}")
print(f"\nSample child:\n  {children[0]['text'][:200]}")
print(f"\nIts parent:\n  {parents[children[0]['parent_id']]['text'][:300]}")

## 3. Technique 2 — Hybrid Search (BM25 + Vector + RRF)

Both indexes operate on **child** chunks. After fusion, we look up the **parent** for generation.

In [ ]:
# --- Vector index on child chunks ---
def get_embeddings(texts: list[str], batch_size: int = 100) -> np.ndarray:
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(input=batch, model=EMBED_MODEL)
        all_emb.extend([item.embedding for item in resp.data])
        print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)}", end="\r")
    print()
    return np.array(all_emb, dtype=np.float32)

print("Embedding child chunks...")
child_texts = [c["text"] for c in children]
child_emb = get_embeddings(child_texts)
faiss.normalize_L2(child_emb)

dim = child_emb.shape[1]
vector_index = faiss.IndexFlatIP(dim)
vector_index.add(child_emb)
print(f"FAISS index: {vector_index.ntotal} child vectors (dim={dim})")

In [ ]:
# --- BM25 index on child chunks ---
def tokenize(text: str) -> list[str]:
    return re.findall(r"[\w]+", text.lower())

tokenized_children = [tokenize(t) for t in child_texts]
bm25 = BM25Okapi(tokenized_children)
print(f"BM25 index built over {len(tokenized_children)} child chunks")

In [ ]:
def rrf_fusion(ranked_lists: list[list[int]], k: int = RRF_K) -> list[int]:
    """Reciprocal Rank Fusion — returns child indices sorted by fused score."""
    scores: dict[int, float] = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


def hybrid_retrieve(query: str, retrieve_k: int = RETRIEVE_K) -> list[dict]:
    """Returns top retrieve_k children after BM25+vector RRF fusion."""
    # Vector search
    q_emb = np.array(
        client.embeddings.create(input=[query], model=EMBED_MODEL).data[0].embedding,
        dtype=np.float32,
    ).reshape(1, -1)
    faiss.normalize_L2(q_emb)
    _, vec_indices = vector_index.search(q_emb, retrieve_k)
    vec_ranked = vec_indices[0].tolist()

    # BM25 search
    scores = bm25.get_scores(tokenize(query))
    bm25_ranked = np.argsort(scores)[::-1][:retrieve_k].tolist()

    # Fuse
    fused = rrf_fusion([vec_ranked, bm25_ranked])[:retrieve_k]
    return [children[i] for i in fused]


print("Hybrid retrieval ready")

## 4. Technique 3 — Cross-Encoder Reranking

After hybrid search returns 20 candidate child chunks, the cross-encoder rescores each `(query, child_text)` pair jointly and we keep the top-5. We then look up the **parent** chunk for each winner.

In [ ]:
print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
reranker = CrossEncoder(RERANKER_MODEL)
print("Cross-encoder loaded")

In [ ]:
def rerank(query: str, candidates: list[dict], top_k: int = TOP_K) -> list[dict]:
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [{"rerank_score": float(s), **c} for s, c in ranked[:top_k]]


def get_parents_for(reranked_children: list[dict]) -> list[dict]:
    """Deduplicate and fetch parent chunks for the winning children."""
    seen, result = set(), []
    for child in reranked_children:
        pid = child["parent_id"]
        if pid not in seen:
            seen.add(pid)
            parent = parents[pid].copy()
            parent["rerank_score"] = child["rerank_score"]
            result.append(parent)
    return result


def run_advanced_rag(question: str) -> tuple[str, list[dict], list[dict]]:
    candidates  = hybrid_retrieve(question)          # 20 child chunks via RRF
    reranked    = rerank(question, candidates)        # top-5 children by cross-encoder
    ctx_parents = get_parents_for(reranked)           # expand to parent chunks
    answer      = generate_answer(question, ctx_parents)
    return answer, reranked, ctx_parents


print("Advanced RAG pipeline ready")

## 5. Generation (same prompt as Naive RAG)

In [ ]:
SYSTEM_PROMPT = """Ты — помощник, отвечающий на вопросы по годовому отчёту компании АО «ЛОТТЕ Рахат».
Отвечай ТОЛЬКО на основе предоставленного контекста. Если информации недостаточно — скажи об этом.
Отвечай кратко и по существу."""

def generate_answer(question: str, ctx_chunks: list[dict]) -> str:
    context = "\n\n---\n\n".join(
        f"[Страница {c['page_num']}]\n{c['text']}" for c in ctx_chunks
    )
    user_msg = f"Контекст:\n{context}\n\nВопрос: {question}"
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

### Quick sanity check

In [ ]:
test_q = "Кто является Председателем правления АО «ЛОТТЕ Рахат»?"
test_answer, test_reranked, test_parents = run_advanced_rag(test_q)

print(f"Q: {test_q}")
print(f"A: {test_answer}")
print(f"\nTop child chunks after reranking (scores):")
for c in test_reranked:
    print(f"  page={c['page_num']}  rerank={c['rerank_score']:.3f}  | {c['text'][:80]}...")
print(f"\nParent chunks sent to LLM: pages {[p['page_num'] for p in test_parents]}")

## 6. Run on full Golden Dataset (30 Q&A)

In [ ]:
with open(QA_PATH, "r", encoding="utf-8") as f:
    questions = json.load(f)

advanced_results = []

for item in questions:
    print(f"[{item['id']:02d}/30] {item['question'][:65]}...")
    answer, reranked, ctx_parents = run_advanced_rag(item["question"])
    advanced_results.append({
        "id":               item["id"],
        "question":         item["question"],
        "ground_truth":     item["ground_truth"],
        "category":         item["category"],
        "source_page":      item["source_page"],
        "predicted_answer": answer,
        "retrieved_chunks": [
            {"page_num": p["page_num"], "rerank_score": p["rerank_score"], "text": p["text"]}
            for p in ctx_parents
        ],
    })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(advanced_results, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(advanced_results)} results to {OUTPUT_PATH}")

## 7. Retrieval metrics

In [ ]:
hits = 0
reciprocal_ranks = []

for r in advanced_results:
    retrieved_pages = [c["page_num"] for c in r["retrieved_chunks"]]
    source = r["source_page"]
    if source in retrieved_pages:
        hits += 1
        rank = retrieved_pages.index(source) + 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

hit_rate = hits / len(advanced_results)
mrr = np.mean(reciprocal_ranks)

print(f"Hit Rate @ {TOP_K}: {hit_rate:.3f}  ({hits}/{len(advanced_results)})")
print(f"MRR:               {mrr:.3f}")

from collections import defaultdict
cat_hits = defaultdict(list)
for r, rr in zip(advanced_results, reciprocal_ranks):
    cat_hits[r["category"]].append(rr > 0)

print("\nHit Rate by category:")
for cat, values in cat_hits.items():
    print(f"  {cat:<12}: {sum(values)}/{len(values)} = {sum(values)/len(values):.3f}")

## 8. Quick results preview

In [ ]:
print(f"{'ID':<4} {'Cat':<10} {'Ground Truth':<45} {'Predicted':<45}")
print("-" * 110)
for r in advanced_results:
    gt  = r["ground_truth"][:43]
    ans = r["predicted_answer"][:43]
    print(f"{r['id']:<4} {r['category']:<10} {gt:<45} {ans:<45}")